# A tiny language model

The deck ended on a network that computes the table of a language model. The corpus $x$ goes in, every row of $x$ becomes a row of numbers in $E$, and out comes $\hat Y$: one row per row of $x$, one column per word of the dictionary, and in each cell the chance that the word of that column comes next. This notebook builds that network for a language of six words and trains it in seconds.

* Every word of the dictionary gets **16 numbers**, its **embedding**, kept in a table, and so does every **place** in a text. Both tables are weights: training moves them, like every other weight of the network.
* **Attention** lets every row read the rows above it in its own text, its prefix.
* The output $\hat Y$ is a **matrix of chances**: every row is a text so far, every column a word, and every number the chance that this word comes next.

The names follow the deck:

| The deck | This notebook |
|---|---|
| the corpus, $x$ and $y$ | `corpus`, `x` and `y`, lists of words |
| the embedding table | `word_table` |
| $E$ | `E` |
| $\hat Y$ | `Y_hat` |
| $\ell_i$ and $\bar\ell$ | `ell` and `ell_bar`, called `loss` in training |

Run the cells in order, or choose Runtime, Run all.

In [ ]:
#@title Imports { display-mode: "form" }
import math
import random
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

## 1. Six words and three rules

The language has six words, `if`, `then`, `either`, `or`, `rain` and `sun`, and three rules:

* `rain` is a sentence, and so is `sun`.
* If A and B are sentences, then `if A then B` is a sentence.
* If A and B are sentences, then `either A or B` is a sentence.

So `if rain then sun` is a sentence, and so is `either sun or if rain then sun`. Every `then` must answer its own `if`, and every `or` its own `either`, however far back they are. In 1957 Noam Chomsky used exactly these two pairs to argue that no machine with a fixed, limited memory, such as one that remembers only the last few words, can produce English. To guess the next word, a model must read the whole prefix.

`make_sentence` follows the three rules and picks at random. The deeper inside a sentence it is, the likelier it picks a plain `rain` or `sun`, so no sentence goes more than three levels deep.

In [ ]:
random.seed(1)                                               # seeded: the same sentences on every computer
PLAIN = [0.1, 0.45, 0.7, 1.0]                                # the chance of a plain rain or sun at depth 0, 1, 2, 3


def make_sentence(depth=0):
    if random.random() < PLAIN[depth]:
        return [random.choice(["rain", "sun"])]                                          # rule 1
    if random.random() < 0.5:
        return ["if"] + make_sentence(depth + 1) + ["then"] + make_sentence(depth + 1)    # rule 2
    return ["either"] + make_sentence(depth + 1) + ["or"] + make_sentence(depth + 1)      # rule 3


for k in range(8):
    print(" ".join(make_sentence()))

In [ ]:
#@title `is_grammatical` checks a list of words against the three rules (double-click to view the code) { display-mode: "form" }
def read_one_sentence(words, i):
    # Reads one sentence that starts at word i. Returns the place where it ends, or -1 if a rule fails.
    if i >= len(words):
        return -1
    if words[i] in ("rain", "sun"):
        return i + 1
    if words[i] in ("if", "either"):
        middle = read_one_sentence(words, i + 1)
        partner = "then" if words[i] == "if" else "or"
        if middle == -1 or middle >= len(words) or words[middle] != partner:
            return -1
        return read_one_sentence(words, middle + 1)
    return -1


def is_grammatical(words):
    return read_one_sentence(words, 0) == len(words)


for text in ["if rain then sun", "either sun or if rain then sun", "if rain or sun", "if rain then"]:
    print(f"{text!r:36} grammatical: {is_grammatical(text.split())}")

## 2. The corpus, $x$ and $y$

The dictionary has seven words: the six of the language and `END`, the word that separates two texts. Here every text is one sentence of the language. As in the deck, the corpus reads 3000 texts one after the other into one column, with `END` before the first text, between every two texts and after the last.

* `x` is the corpus without its last `END`. Every word of `x` gets a row.
* `y` is the shifted corpus: the corpus without its first `END`, moved up one row. `y[i]` is the word that came after `x[i]`.

In [ ]:
WORDS = ["if", "then", "either", "or", "rain", "sun", "END"]   # the dictionary: one column of Y_hat per word

texts = [make_sentence() for k in range(3000)]               # 3000 texts of the language
corpus = ["END"]
for text in texts:
    corpus += text + ["END"]                                 # the words of the text, then END
x = corpus[:-1]                                              # the corpus without its last END
y = corpus[1:]                                               # the shifted corpus: y[i] is the word that came after x[i]
print(len(corpus), "words in the corpus, so x and y have", len(x), "rows each")

The first rows of `x` and `y`, side by side, with the **prefix** of every row. The prefix is read upward from the row to the first `END`, both included: it is the text so far. An `END` row is its own prefix: a text is about to start, and its `y` is the first word of that text. Python counts the rows from 0.

In [ ]:
top = 0                                                      # the END row that the current text starts at
print(" row  x[i]    y[i]    the prefix of row i")
for i in range(16):
    if x[i] == "END":
        top = i                                              # a new text starts at this row
    prefix = " ".join(x[top:i + 1])                          # read upward from row i to the END
    print(f"{i:>4}  {x[i]:<7} {y[i]:<7} {prefix}")

The model reads **one text at a time**, so that every row reads only its own text. The code cuts `x` at every `END` into one piece per text, and `y` at the same rows. The piece of `x` of a text is `END` and its words; the piece of `y` is its words and `END`, row by row the words that the piece of `x` must predict.

In [ ]:
starts = [i for i in range(len(x)) if x[i] == "END"]         # the END rows of x: every text starts at one of them
starts.append(len(x))                                        # so text k fills the rows starts[k] to starts[k + 1] - 1
x_texts = [x[starts[k]:starts[k + 1]] for k in range(len(texts))]   # x cut into texts
y_texts = [y[starts[k]:starts[k + 1]] for k in range(len(texts))]   # y cut at the same rows

print("text 0, its piece of x:", x_texts[0])
print("text 0, its piece of y:", y_texts[0])

## 3. The embeddings: two tables

Every row of `x` becomes a row of numbers in `E`, and two `nn.Embedding` tables hold those numbers. An `nn.Embedding(7, 16)` is a table of 7 rows of 16 numbers: given a list of row numbers, it returns those rows, one under the other.

* The **word table**, the embedding table of the deck: a row of $d = 16$ numbers for each of the 7 words of the dictionary.
* The **place table**: a row of 16 numbers for each place in a text, up to the 40th. The place of a row is its distance from the `END` of its text: 0 for the `END`, 1 for the first word, and so on.

Row $i$ of `E` is the row of its word $x_i$ in the word table **plus** the row of its place in the place table. That is how the model can tell `if rain then` from `then rain if`: the same three rows of the word table, added to different rows of the place table. Both tables start as small random numbers, with a spread of $0.02$, so every word and every place starts close to $0$, and both are weights: training moves them.

A word is known by its place in `WORDS`: `if` is word 0, `then` word 1, and so on up to `END`, word 6. `numbers` turns a list of words into these numbers.

In [ ]:
def numbers(words):
    return torch.tensor([WORDS.index(word) for word in words])   # every word by its place in WORDS


print(numbers(x_texts[0]))

In [ ]:
torch.manual_seed(1)                                         # seeded: the same starting numbers on every computer
d = 16                                                       # numbers per word: E has d columns
LONGEST = 40                                                 # places in the place table

word_table = nn.Embedding(len(WORDS), d)                     # 7 words, 16 numbers each
nn.init.normal_(word_table.weight, std=0.02)                 # every word starts close to 0
place_table = nn.Embedding(LONGEST, d)                       # 40 places, 16 numbers each
nn.init.normal_(place_table.weight, std=0.02)                # every place starts close to 0
word_table_before = word_table.weight.detach().clone()       # kept, to see later how far each word moved

print("word table: ", word_table.weight.shape)
print("place table:", place_table.weight.shape)

In [ ]:
#@title The word table as a picture (double-click to view the code) { display-mode: "form" }
BLUE_WHITE_RED = LinearSegmentedColormap.from_list("blue_white_vermillion", ["#0072B2", "#FFFFFF", "#D55E00"])
BLUES = LinearSegmentedColormap.from_list("white_blue", ["#FFFFFF", "#0072B2"])
CIRCLE = "#D55E00"


def show_tables(tables, titles):
    fig, axes = plt.subplots(1, len(tables), figsize=(6.2 * len(tables), 3.0), squeeze=False)
    for ax, table, title in zip(axes[0], tables, titles):
        picture = ax.imshow(table.detach().numpy(), cmap=BLUE_WHITE_RED, vmin=-1.5, vmax=1.5, aspect="auto")
        ax.set_yticks(range(len(WORDS)), WORDS)
        ax.set_xticks(range(d), [str(k + 1) for k in range(d)])
        ax.set_xlabel("the 16 numbers of each word")
        ax.set_title(title, loc="left")
    fig.colorbar(picture, ax=list(axes[0]), shrink=0.85)
    plt.show()


show_tables([word_table_before], ["The word table before training"])

Before training the picture is blank: every number is close to $0$, and no word is like any other yet.

## 4. Attention: a row reads its prefix

Row $i$ of `E` knows its word and its place, and nothing yet about the words above it. **Attention** is how a row reads the rows above it in its own text. Every row of `E` is turned into a *query*, what it looks for, a *key*, what it offers, and a *value*, what it hands over. Row $i$ gives every row $j$ a score, how well the key of $j$ matches its query; the softmax turns the scores of row $i$ into weights that add up to $1$, and row $i$ reads the weighted average of the values.

A row may read only its **prefix**, itself and the rows above it in its text, since the rows below it hold the words it must guess. Two things see to that. The model reads one text at a time, so the rows of other texts are out of its reach. Inside the text, `no_peeking(n)` adds minus infinity to every score right of the diagonal, the scores of the rows below; after the softmax, a score of minus infinity becomes a weight of $0$.

In [ ]:
#@title `no_peeking(n)`: minus infinity right of the diagonal (double-click to view the code) { display-mode: "form" }
def no_peeking(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)


print(no_peeking(4))

The model: the two tables, the three matrices of attention, and a small network that turns every row into 7 scores, one for each word that could come next. `WQ`, `WK` and `WV` are each an `nn.Linear`, a matrix with its bias, so that training can move them. The scores are divided by $\sqrt d$, which keeps them in a range where the softmax still spreads its weights. Every row keeps its own embedding and adds what it read, `E + A`, before the network.

In [ ]:
WQ = nn.Linear(d, d)                                         # queries: what every row looks for
WK = nn.Linear(d, d)                                         # keys: what every row offers
WV = nn.Linear(d, d)                                         # values: what every row hands over
network = nn.Sequential(
    nn.Linear(d, 64), nn.ReLU(),
    nn.Linear(64, len(WORDS)),                               # out: 7 scores, one per word of the dictionary
)


def next_word_scores(rows):                                  # rows: the piece of x of one text
    n = len(rows)
    E = word_table(numbers(rows)) + place_table(torch.arange(n))   # row i: its word plus its place
    Q, K, V = WQ(E), WK(E), WV(E)                            # queries, keys and values, one row per row of E
    S = (Q @ K.T / d ** 0.5 + no_peeking(n)).softmax(dim=1)  # the attention map: row i reads only its prefix
    A = S @ V                                                # what every row read
    return network(E + A)                                    # n rows, 7 scores each

## 5. The output: the matrix $\hat Y$

The softmax turns every row of 7 scores into 7 chances that add up to $1$: it takes $e$ to the power of every score and divides by their sum. For one text, the result is that text's rows of $\hat Y$. **Every row is the text so far, every column is a word, and every number is the chance that this word comes next.**

In [ ]:
def chances(rows):
    return next_word_scores(rows).softmax(dim=1)             # one row per row of x, 7 chances each


example = "if either rain or sun then sun".split()
x_example = ["END"] + example                                # its piece of x
y_example = example + ["END"]                                # its piece of y
Y_hat_example = chances(x_example)                           # its rows of Y_hat
print(Y_hat_example.shape, "  8 rows, one per row of x; 7 columns, one per word")
print("every row adds up to", Y_hat_example.sum(dim=1).detach())

In [ ]:
#@title The matrix of chances as a table, the word that came next circled (double-click to view the code) { display-mode: "form" }
def show_chances(x_rows, y_rows, title):
    with torch.no_grad():
        table = chances(x_rows).numpy()
    prefixes = [" ".join(x_rows[:i + 1]) for i in range(len(x_rows))]
    fig, ax = plt.subplots(figsize=(9.4, 0.42 * len(prefixes) + 1.3))
    ax.imshow(table, cmap=BLUES, vmin=0, vmax=1, aspect="auto")
    for row in range(table.shape[0]):
        for column in range(table.shape[1]):
            ax.text(column, row, f"{table[row, column]:.2f}", ha="center", va="center", fontsize=10,
                    color="white" if table[row, column] > 0.6 else "black")
        ax.plot(WORDS.index(y_rows[row]), row, "o", markersize=27, markerfacecolor="none",
                markeredgecolor=CIRCLE, markeredgewidth=2)   # the circle: y_i, the word that came next
    ax.set_xticks(range(len(WORDS)), WORDS)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(prefixes)), prefixes)
    ax.set_ylabel("the prefix of the row")
    ax.set_title(title, loc="left", pad=26)
    plt.show()


show_chances(x_example, y_example, "Before training: the chance of every word, after every prefix")

Before training every chance is close to $1/7 = 0.14$: the model has no idea yet which word comes next. The circle in every row marks the entry of the word that came next, $\hat y_{i, y_i}$, the entry the loss reads.

Stacked for all 3000 texts, one piece under the other in the order of `x`, the pieces make the whole matrix $\hat Y$, with one row per row of `x`:

In [ ]:
def all_scores():
    with torch.no_grad():
        return torch.cat([next_word_scores(piece) for piece in x_texts])   # the pieces of all texts, stacked in the order of x


scores = all_scores()                                        # every row of x, 7 scores each
Y_hat = scores.softmax(dim=1)                                # the whole matrix of chances
print("Y_hat:", Y_hat.shape, "  one row per row of x, one column per word")
print(f"every row adds up to 1: the sums lie between {Y_hat.sum(dim=1).min():.6f} and {Y_hat.sum(dim=1).max():.6f}")

## 6. The loss, by hand

The loss of the deck. In every row, take the entry of the word that came next, $\hat y_{i, y_i}$, the circled one, and its minus logarithm:

$$\ell_i = -\log \hat y_{i, y_i}.$$

The loss $\bar\ell$ is the average of the column $\ell$. A row that gave the word that came next a chance of $1$ costs $-\log 1 = 0$; the smaller that chance, the more the row costs. `loss_by_hand` does this for the rows of one text, and prints the column.

In [ ]:
def loss_by_hand(x_rows, y_rows):
    Y_hat_rows = chances(x_rows)
    ell = []                                                 # the loss column, one entry per row
    for i in range(len(x_rows)):
        circled = Y_hat_rows[i, WORDS.index(y_rows[i])].item()   # the chance row i gave the word that came next
        ell.append(math.log(1 / circled))                    # ell_i = -log yhat_{i, y_i} = log(1 / yhat_{i, y_i})
        prefix = " ".join(x_rows[:i + 1])
        print(f"{i:>3}  {prefix:<34} y_i = {y_rows[i]:<6}  yhat = {circled:.3f}   ell_i = {ell[i]:.3f}")
    return sum(ell) / len(ell)                               # ell_bar: the average of the column


ell_bar = loss_by_hand(x_example, y_example)
print()
print(f"ell_bar by hand:              {ell_bar:.4f}")
print(f"nn.functional.cross_entropy:  {nn.functional.cross_entropy(next_word_scores(x_example), numbers(y_example)).item():.4f}")

`nn.functional.cross_entropy` takes the scores before the softmax and the words that came next, and does the softmax, the minus logarithm and the average itself: it gives the same number. Before training every $\ell_i$ is close to $\log 7 = 1.95$, and so is $\bar\ell$ on the whole corpus:

In [ ]:
print(f"the loss on the whole corpus: {nn.functional.cross_entropy(scores, numbers(y)).item():.4f}")

## 7. Training

Training makes $\bar\ell$ small, 32 texts at a time.

* A step picks 32 texts at random. The model reads the piece of `x` of each text and gives its rows of $\hat Y$; the step stacks the 32 pieces one under the other, and their pieces of `y` in the same order, so that row $r$ of the two stacks belongs to the same row of `x`.
* `nn.functional.cross_entropy` gives $\bar\ell$ of the stacked rows, the `loss`.
* `loss.backward()` computes the gradient of the loss for every weight, and `optimizer.step()` moves them all: the weights of the network and of `WQ`, `WK` and `WV`, and **also the 16 numbers of every word and of every place**, since the optimizer lists the two tables beside the network. Adam is gradient descent with a step size that it adapts for every weight.
* The loss of 32 texts jumps around from step to step, so the loop watches its **running average**: every step keeps 0.9 of the average and adds 0.1 of the new loss, which smooths out the jumps.

The loss cannot reach $0$. Wherever a new sentence can start, at the top of a text and right after `if`, `then`, `either` or `or`, the rules allow `if`, `either`, `rain` and `sun` (only the last two at the deepest level), and chance decides between them. Even a model that knew `make_sentence` by heart would score about $0.73$. The loop stops when the running average falls below $0.74$.

In [ ]:
optimizer = torch.optim.Adam(list(word_table.parameters()) + list(place_table.parameters())
                             + list(WQ.parameters()) + list(WK.parameters()) + list(WV.parameters())
                             + list(network.parameters()), lr=0.01)

running_loss = 2.0                                           # the running average of the loss, any start above 0.74
losses = []                                                  # the loss of every step, kept for the plot below
running_losses = []                                          # and the running average after every step
step = 0
while running_loss > 0.74:
    pick = torch.randint(0, len(texts), (32,))               # 32 texts picked at random
    scores = torch.cat([next_word_scores(x_texts[k]) for k in pick])   # their rows of Y_hat, stacked, before the softmax
    targets = torch.cat([numbers(y_texts[k]) for k in pick])           # their rows of y, stacked in the same order
    loss = nn.functional.cross_entropy(scores, targets)      # ell_bar: the average of -log yhat_{i, y_i} over the rows
    optimizer.zero_grad()                                    # forget the gradients of the previous step
    loss.backward()                                          # compute the gradient of the loss
    optimizer.step()                                         # move every weight, every word and every place
    running_loss = 0.9 * running_loss + 0.1 * loss.item()    # the running average
    losses.append(loss.item())
    running_losses.append(running_loss)
    step += 1
    print(f"\rstep {step}, running average of the loss: {running_loss:.4f}", end="")
print(f"\nstopped after {step} steps")

The loss of every step, and its running average. The loss of 32 texts jumps around; the running average falls fast, then flattens out just above the floor:

In [ ]:
#@title The loss during training (double-click to view the code) { display-mode: "form" }
steps = range(1, len(losses) + 1)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.plot(steps, losses, color="#AAAAAA", linewidth=0.8, label="the loss of the step")
ax.plot(steps, running_losses, color="#0072B2", linewidth=2, label="its running average")
ax.axhline(0.74, color="black", linestyle=":", linewidth=1, label="the loop stops below 0.74")
ax.axhline(0.73, color="#D55E00", linestyle="--", linewidth=1, label="the floor, about 0.73")
ax.set_xlabel("step")
ax.set_ylabel("the loss")
ax.set_ylim(0, 2.2)
ax.legend()
plt.show()

## 8. The matrix after training

The same example, and its loss column:

In [ ]:
show_chances(x_example, y_example, "After training: the chance of every word, after every prefix")

In [ ]:
print(f"\nell_bar of the example: {loss_by_hand(x_example, y_example):.4f}")

What to look for:

* Where the rules leave a choice, the model spreads its chances over the words the rules allow: at the top of a text, after the prefix `END`, a sentence can open with any of `if`, `either`, `rain` and `sun`, and after `END if` as well. These rows keep a loss above $0$.
* It even learns how often each choice comes: `make_sentence` opens a text with `if` or `either` 45 times in 100 each, and with `rain` or `sun` 5 times each, and the first row reads close to that.
* Where the rules force a word, the model puts nearly everything on it: after `END if either rain` only `or` can come, and after `END if either rain or sun` only `then`, the answer to the `if` four words back. After the last `sun` the text is complete, and `END` must come. These rows cost almost nothing.
* The words the rules forbid get almost nothing.

On the whole corpus, the loss ends close to the floor of $0.73$:

In [ ]:
scores = all_scores()                                        # every row of x, after training
Y_hat = scores.softmax(dim=1)
print("Y_hat:", Y_hat.shape)
print(f"the loss on the whole corpus: {nn.functional.cross_entropy(scores, numbers(y)).item():.4f}")

## 9. The word table after training

In [ ]:
#@title Before and after, and every word's nearest word (double-click to view the code) { display-mode: "form" }
show_tables([word_table_before, word_table.weight], ["Before training", "After training"])
after = word_table.weight.detach()
for word in WORDS:
    distances = [(after[WORDS.index(word)] - after[WORDS.index(other)]).norm().item() for other in WORDS]
    others = [(distance, other) for distance, other in zip(distances, WORDS) if other != word]
    distance, nearest = min(others)
    print(f"{word:>7}: nearest word {nearest!r:9} at a distance of {distance:.2f}")

Nobody told the model that `rain` and `sun` are the same kind of word. The rules use them the same way, so training gave them the same push at every step, and they ended up with nearly the same 16 numbers. The other words keep further apart, because the model needs to tell them apart: `if` and `either` both open a sentence, but they wait for different partners.

## 10. The model writes

`write_text` starts from the prefix `END`, a text about to start. It takes the last row of the text's $\hat Y$, the chances after the text so far, picks the next word with those 7 chances, like a spin of the deck's roulette, and goes on until it picks `END`.

In [ ]:
def write_text():
    rows = ["END"]                                           # the prefix END: a text is about to start
    next_word = ""
    while next_word != "END" and len(rows) < LONGEST:
        with torch.no_grad():
            row = chances(rows)[-1]                          # the last row: the chances after the text so far
        next_word = random.choices(WORDS, weights=row.tolist())[0]   # one word, picked with these 7 chances
        rows.append(next_word)
    return [word for word in rows if word != "END"]          # the text, without its ENDs


for k in range(10):
    text = write_text()
    print("yes" if is_grammatical(text) else "NO ", " ".join(text))

In [ ]:
made = [write_text() for k in range(1000)]
grammatical = [text for text in made if is_grammatical(text)]
new = [text for text in grammatical if text not in texts]
print(len(grammatical), "of 1000 texts follow the rules, and", len(new), "of them are not among the 3000 it learned from")

## 11. Where the rows look

The attention map `S` of the example after training, one row per row of its piece of `x`: how much the row reads itself and every row above it. Above the diagonal every weight is $0$: a row reads only its prefix.

Look at the rows where the rules force the next word. In our run, row 3, `rain`, is where the model must see that `or` comes next, and it reads its `either`, row 2, with a weight of 0.91. Row 5, `sun`, is where `then` is due, and it reads the `if` of row 1, four rows up, with a weight of 0.82. Each of these rows finds the word that opened its pair: this is what Chomsky's argument asks of a model. With a single layer of attention the map depends on the run, and on another computer its numbers can come out different, while the chances stay close to the rules.

In [ ]:
#@title The attention map of the example (double-click to view the code) { display-mode: "form" }
def attention_map(rows):
    n = len(rows)
    with torch.no_grad():
        E = word_table(numbers(rows)) + place_table(torch.arange(n))
        return (WQ(E) @ WK(E).T / d ** 0.5 + no_peeking(n)).softmax(dim=1)


S = attention_map(x_example).numpy()
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.imshow(S, cmap=BLUES, vmin=0, vmax=1)
for row in range(len(x_example)):
    for column in range(row + 1):
        ax.text(column, row, f"{S[row, column]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if S[row, column] > 0.6 else "black")
ax.set_xticks(range(len(x_example)), x_example)
ax.xaxis.tick_top()
ax.set_yticks(range(len(x_example)), [f"{i}. {word}" for i, word in enumerate(x_example)])
ax.set_xlabel("reads the row of")
plt.show()